# RFM Customer Segmentation - Batch Scoring
### Based on: Wong, Tong & Haw (2024) - Exploring Customer Segmentation in E-Commerce using RFM Analysis with Clustering Techniques
### Journal: Journal of Telecommunications and the Digital Economy, Vol. 12 No. 3, pp. 97–125
### DOI: https://doi.org/10.18080/jtde.v12n3.978

In [0]:
import os
import sys
import io

import warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import PowerTransformer
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

mlflow.autolog(disable=True)

print("imports done")

**Load Model and Scaler from MLflow**

In [0]:
model_name = "bharatmart.ml.bharatmart_rfm_model"

km_model = mlflow.sklearn.load_model(f"models:/{model_name}@Production")

print(f"model loaded : {model_name}@Production")
print(f"n_clusters : {km_model.n_clusters}")

**Load Scaler from MLflow**

In [0]:
client = MlflowClient()
model_info = client.get_model_version_by_alias(model_name, "Production")
run_id = model_info.run_id

scaler_path = mlflow.artifacts.download_artifacts(
    run_id = run_id,
    artifact_path = "scaler/rfm_scaler.pkl"
)

with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

print(f"scaler loaded from run : {run_id}")
print(f"scaler type : {type(scaler)}")

**Load RFM Features**

In [0]:
churn_features = spark.table("bharatmart.ml.churn_features")

rfm_df = churn_features.select(
    "customer_id", "recency_days", "total_orders", "total_spent"
)

print(f"rows : {rfm_df.count():,}")

In [0]:
display(rfm_df.limit(5))

**Clip and Normalise**

In [0]:
rfm_pdf = rfm_df.toPandas()
rfm_pdf["total_orders"] = rfm_pdf["total_orders"].astype(float)
rfm_pdf["total_spent"]  = rfm_pdf["total_spent"].astype(float)

# compute p99 fresh from today's data
p99_spent = float(np.percentile(rfm_pdf["total_spent"], 99))
rfm_pdf["total_spent"] = rfm_pdf["total_spent"].clip(upper=p99_spent)

print(f"p99 today = Rs. {p99_spent:,.0f}")
print(f"max after clip = Rs. {rfm_pdf['total_spent'].max():,.0f}")

**Apply Saved Scaler**

In [0]:
rfm_features = rfm_pdf[["recency_days", "total_orders", "total_spent"]].copy()

rfm_scaled = scaler.transform(rfm_features)

rfm_scaled_df = pd.DataFrame(
    rfm_scaled,
    columns=["recency_scaled", "orders_scaled", "spent_scaled"]
)

print(f"shape : {rfm_scaled_df.shape}")
print(f"means : {rfm_scaled_df.mean().round(3).to_dict()}")

**Assign Clusters**

In [0]:
cluster_labels = km_model.predict(rfm_scaled_df)

rfm_pdf["rfm_cluster_id"] = cluster_labels

segment_map = {
    1: "Champions",
    0: "Loyal Customers",
    2: "At Risk"
}

rfm_pdf["rfm_segment"] = rfm_pdf["rfm_cluster_id"].map(segment_map)

print(rfm_pdf["rfm_segment"].value_counts())

**Join Churn Probability**

In [0]:
churn_preds = spark.table("bharatmart.ml.gld_churn_predictions") \
    .select("customer_id", "churn_probability")

churn_pdf = churn_preds.toPandas()

rfm_pdf = rfm_pdf.merge(churn_pdf, on="customer_id", how="left")

print(f"rows after join : {len(rfm_pdf):,}")
print(f"churn_prob nulls : {rfm_pdf['churn_probability'].isnull().sum()}")

**Compute Retention Priority**

In [0]:
def retention_priority(segment, churn_prob):
    if segment == "Champions" and churn_prob >= 0.5:
        return "Urgent"
    elif segment == "Champions":
        return "High"
    elif segment == "Loyal Customers" and churn_prob >= 0.5:
        return "High"
    elif segment == "Loyal Customers":
        return "Medium"
    else:
        return "Low"

rfm_pdf["retention_priority"] = rfm_pdf.apply(
    lambda row: retention_priority(row["rfm_segment"], row["churn_probability"]),
    axis=1
)

print(rfm_pdf["retention_priority"].value_counts())

**Write gld_rfm_segments**

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType
from datetime import datetime

rfm_pdf["algorithm_used"]    = "KMeans"
rfm_pdf["silhouette_score"]  = 0.4108
rfm_pdf["segmented_date"]    = datetime.now()

output_cols = [
    "customer_id", "rfm_segment", "rfm_cluster_id",
    "recency_days", "total_orders", "total_spent",
    "algorithm_used", "silhouette_score",
    "churn_probability", "retention_priority", "segmented_date"
]

output_df = spark.createDataFrame(rfm_pdf[output_cols])

output_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bharatmart.ml.gld_rfm_segments")

print(f"gld_rfm_segments written : {output_df.count():,} rows")

**Verify Output Table**

In [0]:
verify = spark.table("bharatmart.ml.gld_rfm_segments")

print(f"rows : {verify.count():,}")
print(f"columns : {len(verify.columns)}")
print()
print(verify.groupBy("rfm_segment", "retention_priority") \
    .count() \
    .orderBy("rfm_segment", "retention_priority") \
    .toPandas().to_string(index=False))